In [1]:
import pandas as pd
import numpy as np

# Cargar datasets originales
df_vuelos = pd.read_csv('../data/raw/TraficoAereoMensual2022-2026.csv')
df_kpi = pd.read_csv('../data/raw/kpi_oferta_demanda.csv')

print(f"Dimensiones iniciales Vuelos: {df_vuelos.shape}")
print(f"Dimensiones iniciales KPI: {df_kpi.shape}")

print("\nDuplicados detectados en vuelos:", df_vuelos.duplicated().sum())
print("Duplicados detectados en KPI:", df_kpi.duplicated().sum())

Dimensiones iniciales Vuelos: (24629, 26)
Dimensiones iniciales KPI: (3050, 9)

Duplicados detectados en vuelos: 0
Duplicados detectados en KPI: 0


In [2]:
# Eliminar duplicados exactos
df_vuelos = df_vuelos.drop_duplicates()
df_kpi = df_kpi.drop_duplicates()

# Filtrar operaciones comerciales con pasajeros válidos (> 0)
df_vuelos_limpio = df_vuelos[
    (df_vuelos['PASAJEROS'] > 0) & (df_vuelos['PASAJEROS'].notna())
].copy()

# Crear llave temporal continua YYYYMM en formato numérico entero
df_vuelos_limpio['month_id'] = (
    df_vuelos_limpio['Ano'].astype(str)
    + df_vuelos_limpio['Mes'].astype(str).str.zfill(2)
).astype(int)

print(f"Vuelos comerciales con pasajeros válidos: {len(df_vuelos_limpio):,}")
print(f"Registros KPI disponibles: {len(df_kpi):,}")

Vuelos comerciales con pasajeros válidos: 19,912
Registros KPI disponibles: 3,050


In [3]:
# Normalizar texto base para evitar fallos por espacios o mayúsculas/minúsculas
df_vuelos_limpio['Grupo_norm'] = (
    df_vuelos_limpio['Grupo'].astype(str).str.strip().str.upper()
)
df_kpi['aerolinea_oaci'] = (
    df_kpi['aerolinea_oaci'].astype(str).str.strip().str.upper()
)
df_kpi['inter_dom'] = df_kpi['inter_dom'].astype(str).str.strip().str.upper()

# Diccionario completo: tanto operadores locales como internacionales
mapa_grupo_oaci = {
    # Domésticos y regionales
    'LATAM': 'LAN',
    'SKY': 'SKU',
    'JETSMART': 'JAT',
    'DAP': 'DAP',
    # Internacionales regulares
    'AEROLINEAS ARGENTINAS': 'ARG',
    'AEROMÉXICO': 'AMX',
    'AIR CANADA': 'ACA',
    'AIR FRANCE': 'AFR',
    'AMASZONAS': 'AZN',
    'AMERICAN AIRLINES': 'AAL',
    'ARAJET': 'DWI',
    'AVIANCA': 'AVA',
    'BOLIVIANA DE AVIACIÓN': 'BOV',
    'BRITISH AIRWAYS': 'BAW',
    'CONVIASA': 'VCV',
    'COPA': 'CMP',
    'CUBANA DE AVIACION': 'CUB',
    'DELTA AIRLINES': 'DAL',
    'ESTELAR': 'ETR',
    'GOL': 'GLO',
    'IBERIA': 'IBE',
    'KLM': 'KLM',
    'KOREAN AIR': 'KAL',
    'LEVEL': 'BOS',
    'PARANAIR': 'PRN',
    'QANTAS AIRWAYS': 'QFA',
    'STAR UP': 'SRU',
    'TURKISH AIRLINES': 'THY',
    'UNITED AIRLINES': 'UAL'
}

df_vuelos_limpio['aerolinea_oaci'] = df_vuelos_limpio['Grupo_norm'].map(mapa_grupo_oaci)

# Homologar tráfico para el cruce de KPIs (D = Doméstico, I = Internacional)
nac_a_kpi = {
    'NACIONAL': 'D',
    'INTERNACIONAL': 'I',
    'NAC': 'D',
    'INT': 'I'
}
df_vuelos_limpio['inter_dom'] = (
    df_vuelos_limpio['NAC'].astype(str).str.strip().str.upper().map(nac_a_kpi)
)

# Homologar texto para Streamlit
df_vuelos_limpio['NAC'] = df_vuelos_limpio['NAC'].replace({
    'NACIONAL': 'Nacional',
    'NAC': 'Nacional',
    'INTERNACIONAL': 'Internacional',
    'INT': 'Internacional'
})

# Auditoría: verificar qué operadores quedaron sin código OACI
sin_oaci = sorted(
    df_vuelos_limpio.loc[df_vuelos_limpio['aerolinea_oaci'].isna(), 'Grupo_norm'].unique()
)
print(f"Cantidad de operadores sin código OACI asignado: {len(sin_oaci)}")
if len(sin_oaci) > 0:
    print("Operadores no mapeados:", sin_oaci)

Cantidad de operadores sin código OACI asignado: 0


In [4]:
# Filtrar sólo observaciones con clave OACI y tipo de tráfico válidos para el cruce
df_vuelos_con_oaci = df_vuelos_limpio[
    df_vuelos_limpio['aerolinea_oaci'].notna()
    & df_vuelos_limpio['inter_dom'].notna()
]

# Agregar el tráfico al mismo nivel de observación del archivo KPI
df_trafico_aerolinea = df_vuelos_con_oaci.groupby(
    ['month_id', 'aerolinea_oaci', 'inter_dom'],
    as_index=False
).agg(
    pasajeros=('PASAJEROS', 'sum'),
    rutas=('ORIG_1', 'nunique'),
    destinos=('DEST_1', 'nunique')
)

# Garantizar unicidad de claves en el dataset de KPIs
df_kpi_unico = df_kpi.drop_duplicates(
    subset=['month_id', 'aerolinea_oaci', 'inter_dom']
)

# Cruce con validación one_to_one
df_aerolinea_kpi = pd.merge(
    df_trafico_aerolinea,
    df_kpi_unico,
    on=['month_id', 'aerolinea_oaci', 'inter_dom'],
    how='inner',
    validate='one_to_one'
)

print(f"Tráfico agregado por aerolínea/mes: {df_trafico_aerolinea.shape}")
print(f"Dataset consolidado final de KPI: {df_aerolinea_kpi.shape}")

Tráfico agregado por aerolínea/mes: (1229, 6)
Dataset consolidado final de KPI: (416, 12)


In [5]:
# Eliminar columna temporal de apoyo
if 'Grupo_norm' in df_vuelos_limpio.columns:
    df_vuelos_limpio = df_vuelos_limpio.drop(columns=['Grupo_norm'])

# Guardar base principal (rutas, operadores y pasajeros)
df_vuelos_limpio.to_csv(
    '../data/processed/datos_limpios.csv',
    index=False
)

# Guardar base agregada de flota / indicadores JAC
df_aerolinea_kpi.to_csv(
    '../data/processed/datos_aerolinea_kpi.csv',
    index=False
)

print("Exportación finalizada con éxito:")
print(f"  -> datos_limpios.csv ({len(df_vuelos_limpio):,} filas)")
print(f"  -> datos_aerolinea_kpi.csv ({len(df_aerolinea_kpi):,} filas)")

Exportación finalizada con éxito:
  -> datos_limpios.csv (19,912 filas)
  -> datos_aerolinea_kpi.csv (416 filas)
